In [ ]:
pip install tensorflow category_encoders

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import tqdm

In [13]:
def train_validate_test_split(df, train, validate):
    target_col = "AWARD_VALUE_EURO_FIN_1"
    #df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df)) + train_indice
    print(train_indice, validate_indice)

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice:]

    X_train = train_set.drop(columns = [target_col]).values
    y_train = train_set[target_col].values

    X_val = val_set.drop(columns = [target_col]).values
    y_val = val_set[target_col].values

    X_test = test_set.drop(columns = [target_col]).values
    y_test = test_set[target_col].values

    return X_train, y_train, X_val, y_val, X_test, y_test

In [14]:
def encode_data(df, target_column="award_contract/val_total"):

    base_n_encoder_cols = ["ISO_COUNTRY_CODE", "MAIN_ACTIVITY", "CPV"]
    one_hot_cols = ["CAE_TYPE", "B_ON_BEHALF", "B_AWARDED_BY_CENTRAL_BODY", "TYPE_OF_CONTRACT",
                    "B_FRA_AGREEMENT", "B_EU_FUNDS", "TOP_TYPE", "CRIT_CODE", "lots"]

    encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)
    df = encoder.fit_transform(df)
    df = pd.get_dummies(df, columns=(one_hot_cols), drop_first=True, dtype=int)

    return df

In [15]:
def filter_outliers(df, upper, lower, target_col = "AWARD_VALUE_EURO_FIN_1"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers, with high amount = {}, and low amount = {}".format(len(outlier_indices), high_amount, low_amount))
    df = df.drop(labels = outlier_indices, axis = 0)
    return df

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [17]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_pickle("/content/drive/MyDrive/Thesis/final_data_set_from_csv.pickle")
crit_price_weight_empty = list(df.loc[pd.isna(df["CRIT_PRICE_WEIGHT"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)
df.head()

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df, 0.6, 0.2)

In [21]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
X_train.shape

In [ ]:
class PyTorchModel(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2):
        super(PyTorchModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size1)
        self.regression_layer = nn.Linear(hidden_size1, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        regression_output = self.regression_layer(x)
        return regression_output

# Instantiate your PyTorch model
input_size = X_train[0].shape[0] # Update with your actual input dimension
hidden_size1 = 32
hidden_size2 = 8

pytorch_model = PyTorchModel(input_size, hidden_size1, hidden_size2)

# You can print the model summary if needed
#print(pytorch_model)

# Assuming X_train and y_train are PyTorch tensors
criterion = nn.MSELoss()
optimizer = optim.Adam(pytorch_model.parameters(), lr=0.001)
epochs = 50
# Training loop
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = pytorch_model(X_train)
    y_train_squeezed = y_train.unsqueeze(1)
    loss = criterion(outputs, y_train_squeezed)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item()}")

# After training, you can use the PyTorch model for predictions
with torch.no_grad():
    pytorch_model.eval()
    y_pred_pytorch = pytorch_model(X_test)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([332771, 1, 1])) that is different to the input size (torch.Size([332771, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Assuming X_train, y_train, X_test, y_test are PyTorch tensors
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Instantiate your PyTorch model, optimizer, and loss function
model = nn.Sequential(
    nn.Linear(43, 32),
    nn.ReLU(),
    nn.Linear(32, 8),
    nn.ReLU(),
    nn.Linear(8, 32),
    nn.Linear(32, 1)
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.L1Loss()  # Use L1Loss for MAE

best_mae = float('inf')
best_weights = None
history = []
n_epochs = 5

# Training loop
for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch.unsqueeze(1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Average loss over batches
    avg_loss = total_loss / len(train_loader)

    # Evaluate on the test set
    model.eval()
    with torch.no_grad():
        y_pred_test = model(X_test)
        mae = criterion(y_pred_test, y_test.unsqueeze(1)).item()
        history.append(mae)

    # Save best model
    if mae < best_mae:
        best_mae = mae
        best_weights = model.state_dict()

    # Print training progress
    print(f"Epoch {epoch + 1}/{n_epochs}, Avg. Training Loss: {avg_loss:.4f}, Test MAE: {mae:.4f}")

# Restore the model with the best weights
model.load_state_dict(best_weights)


In [ ]:
history